# 02 - Feature Engineering com MathFeature (PROTEÍNAS)

Extração de características de sequências de **PROTEÍNAS** (aminoácidos) usando a biblioteca MathFeature.

**IMPORTANTE:** As sequências são de aminoácidos, não DNA! Usaremos métodos específicos para proteínas.

In [56]:
from pathlib import Path
from Bio import SeqIO
import pandas as pd
import numpy as np
import subprocess

DATA_RAW = Path("../data/raw")
DATA_PROC = Path("../data/processed")
DATA_PROC.mkdir(parents=True, exist_ok=True)

print("Imports OK")

Imports OK


## 1. Verificar MathFeature

In [57]:
import sys

MATHFEATURE_DIR = Path("/home/jonathan/code/desafio_CD_Bioinfo/MathFeature")
sys.path.insert(0, str(MATHFEATURE_DIR))

# Lista os métodos disponíveis
methods_dir = MATHFEATURE_DIR / "methods"
methods = list(methods_dir.glob("*.py"))
print(f"{len(methods)} métodos encontrados:\n")
for m in sorted(methods):
    print(f"  - {m.stem}")

18 métodos encontrados:

  - AAindexClass
  - AccumulatedNucleotideFrequency
  - ChaosGameTheory
  - CodingClass
  - ComplexNetworksClass-Fast
  - ComplexNetworksClass-v2
  - ComplexNetworksClass
  - EntropyClass
  - ExtractionTechniques-Protein
  - ExtractionTechniques
  - FickettScore
  - FourierClass
  - Kgap
  - MappingClass
  - Mappings-Protein
  - PseKNC
  - TsallisEntropy
  - k-mers


## 2. Preparar dados - Combinar Pos e Neg

In [58]:
# Carrega os arquivos
pos_records = list(SeqIO.parse(DATA_RAW / "Pos_train_fasta.txt", "fasta"))
neg_records = list(SeqIO.parse(DATA_RAW / "Neg_train_fasta.txt", "fasta"))

print(f"Positivos: {len(pos_records)}")
print(f"Negativos: {len(neg_records)}")

# Combina todos os records
all_train = pos_records + neg_records

# Salva em um único arquivo
train_combined = DATA_PROC / "train_combined.fasta"
SeqIO.write(all_train, train_combined, "fasta")

# Cria arquivo de labels separado
labels = [1] * len(pos_records) + [0] * len(neg_records)
ids = [rec.id for rec in all_train]

labels_df = pd.DataFrame({"id": ids, "label": labels})
labels_df.to_csv(DATA_PROC / "train_labels.csv", index=False)

print(f"\nArquivo combinado: {train_combined}")
print(f"Labels salvos: {DATA_PROC / 'train_labels.csv'}")
print(f"Total: {len(all_train)} sequências")

Positivos: 1940
Negativos: 1940

Arquivo combinado: ../data/processed/train_combined.fasta
Labels salvos: ../data/processed/train_labels.csv
Total: 3880 sequências


## 3. Extrair Features de PROTEÍNAS

Vamos usar métodos específicos para sequências de aminoácidos:
1. **AAC, DPC, TPC** - Composição de aminoácidos (ExtractionTechniques-Protein)
2. **Mappings-Protein** - Mapeamentos numéricos + Fourier
3. **Entropy** - Entropia de Shannon (funciona com qualquer alfabeto)

In [59]:
# === AAC - AMINO ACID COMPOSITION ===

print("\n" + "="*60)
print("EXTRAINDO AAC (Amino Acid Composition)")
print("="*60)

train_fasta = DATA_PROC / "train_combined.fasta"
test_fasta = DATA_RAW / "seqs_test.txt"

# TREINO
output_aac_train = DATA_PROC / "features_aac_train.csv"
if output_aac_train.exists():
    output_aac_train.unlink()

cmd_aac_train = [
    "python", str(MATHFEATURE_DIR / "methods" / "ExtractionTechniques-Protein.py"),
    "-i", str(train_fasta),
    "-o", str(output_aac_train),
    "-l", "train",
    "-t", "AAC"
]

print("\nExtraindo AAC do TREINO...")
result_aac_train = subprocess.run(cmd_aac_train, capture_output=True, text=True)

if result_aac_train.returncode == 0:
    df_aac_train = pd.read_csv(output_aac_train)
    print(f"Shape: {df_aac_train.shape}")
    print(f"Features: {df_aac_train.shape[1] - 2}")  # -2 para nameseq e label
else:
    print(f"Erro:\n{result_aac_train.stderr}")

# TEST
output_aac_test = DATA_PROC / "features_aac_test.csv"
if output_aac_test.exists():
    output_aac_test.unlink()

cmd_aac_test = [
    "python", str(MATHFEATURE_DIR / "methods" / "ExtractionTechniques-Protein.py"),
    "-i", str(test_fasta),
    "-o", str(output_aac_test),
    "-l", "test",
    "-t", "AAC"
]

print("\nExtraindo AAC do TEST...")
result_aac_test = subprocess.run(cmd_aac_test, capture_output=True, text=True)

if result_aac_test.returncode == 0:
    df_aac_test = pd.read_csv(output_aac_test)
    print(f"Shape: {df_aac_test.shape}")
else:
    print(f"Erro:\n{result_aac_test.stderr}")


EXTRAINDO AAC (Amino Acid Composition)

Extraindo AAC do TREINO...
Shape: (3880, 22)
Features: 20

Extraindo AAC do TEST...
Shape: (3880, 22)
Features: 20

Extraindo AAC do TEST...
Shape: (970, 22)
Shape: (970, 22)


## 4. Juntar AAC com labels e adicionar DPC

In [60]:
# === JUNTAR AAC COM LABELS ===

labels_df = pd.read_csv(DATA_PROC / "train_labels.csv")

# Merge usando nameseq (features) com id (labels)
df_train = df_aac_train.merge(labels_df, left_on="nameseq", right_on="id", how="left")

# Remove colunas duplicadas
df_train = df_train.drop(columns=["label_x", "id"])
df_train = df_train.rename(columns={"label_y": "label"})

print(f"Dataset treino com AAC: {df_train.shape}")
print(f"Features AAC: {df_train.shape[1] - 2}")  # -2 para nameseq e label
print(f"\nDistribuição de classes:")
print(df_train['label'].value_counts())

# Salva
df_train.to_csv(DATA_PROC / "train_features.csv", index=False)

# Test (sem labels)
df_test = df_aac_test.drop(columns=['label'], errors='ignore')
df_test.to_csv(DATA_PROC / "test_features.csv", index=False)
print(f"\nTest: {df_test.shape}")

Dataset treino com AAC: (3880, 22)
Features AAC: 20

Distribuição de classes:
label
1    1940
0    1940
Name: count, dtype: int64

Test: (970, 21)


## 5. Expandir Features - DPC (Dipeptide Composition)

In [61]:
# === DPC - DIPEPTIDE COMPOSITION ===

print("\n" + "="*60)
print("EXTRAINDO DPC (Dipeptide Composition)")
print("="*60)

# TREINO
output_dpc_train = DATA_PROC / "features_dpc_train.csv"
if output_dpc_train.exists():
    output_dpc_train.unlink()

cmd_dpc_train = [
    "python", str(MATHFEATURE_DIR / "methods" / "ExtractionTechniques-Protein.py"),
    "-i", str(train_fasta),
    "-o", str(output_dpc_train),
    "-l", "train",
    "-t", "DPC"
]

print("\nExtraindo DPC do TREINO...")
result_dpc_train = subprocess.run(cmd_dpc_train, capture_output=True, text=True)

if result_dpc_train.returncode == 0:
    df_dpc_train = pd.read_csv(output_dpc_train)
    print(f"Shape: {df_dpc_train.shape}")
    print(f"Features: {df_dpc_train.shape[1] - 2}")
else:
    print(f"Erro:\n{result_dpc_train.stderr}")

# TEST
output_dpc_test = DATA_PROC / "features_dpc_test.csv"
if output_dpc_test.exists():
    output_dpc_test.unlink()

cmd_dpc_test = [
    "python", str(MATHFEATURE_DIR / "methods" / "ExtractionTechniques-Protein.py"),
    "-i", str(test_fasta),
    "-o", str(output_dpc_test),
    "-l", "test",
    "-t", "DPC"
]

print("\nExtraindo DPC do TEST...")
result_dpc_test = subprocess.run(cmd_dpc_test, capture_output=True, text=True)

if result_dpc_test.returncode == 0:
    df_dpc_test = pd.read_csv(output_dpc_test)
    print(f"Shape: {df_dpc_test.shape}")
else:
    print(f"Erro:\n{result_dpc_test.stderr}")


EXTRAINDO DPC (Dipeptide Composition)

Extraindo DPC do TREINO...
Shape: (3880, 402)
Features: 400

Extraindo DPC do TEST...
Shape: (3880, 402)
Features: 400

Extraindo DPC do TEST...
Shape: (970, 402)
Shape: (970, 402)


## 6. Expandir Features - Entropy Shannon

In [62]:
# === ENTROPY SHANNON (funciona com proteínas) ===

print("\n" + "="*60)
print("EXTRAINDO ENTROPY (Shannon)")
print("="*60)

# TREINO
output_entropy_train = DATA_PROC / "features_entropy_train.csv"
if output_entropy_train.exists():
    output_entropy_train.unlink()

cmd_entropy_train = [
    "python", str(MATHFEATURE_DIR / "methods" / "EntropyClass.py"),
    "-i", str(train_fasta),
    "-o", str(output_entropy_train),
    "-l", "train",
    "-k", "3",  # k-mer size
    "-e", "Shannon"  # tipo de entropia
]

print("\nExtraindo Entropy do TREINO...")
result_entropy_train = subprocess.run(cmd_entropy_train, capture_output=True, text=True)

if result_entropy_train.returncode == 0:
    df_entropy_train = pd.read_csv(output_entropy_train)
    print(f"Shape: {df_entropy_train.shape}")
else:
    print(f"Erro:\n{result_entropy_train.stderr}")

# TEST
output_entropy_test = DATA_PROC / "features_entropy_test.csv"
if output_entropy_test.exists():
    output_entropy_test.unlink()

cmd_entropy_test = [
    "python", str(MATHFEATURE_DIR / "methods" / "EntropyClass.py"),
    "-i", str(test_fasta),
    "-o", str(output_entropy_test),
    "-l", "test",
    "-k", "3",
    "-e", "Shannon"
]

print("\nExtraindo Entropy do TEST...")
result_entropy_test = subprocess.run(cmd_entropy_test, capture_output=True, text=True)

if result_entropy_test.returncode == 0:
    df_entropy_test = pd.read_csv(output_entropy_test)
    print(f"Shape: {df_entropy_test.shape}")
else:
    print(f"Erro:\n{result_entropy_test.stderr}")


EXTRAINDO ENTROPY (Shannon)

Extraindo Entropy do TREINO...
Shape: (3880, 5)

Extraindo Entropy do TEST...
Shape: (3880, 5)

Extraindo Entropy do TEST...
Shape: (970, 5)
Shape: (970, 5)


## 8. Combinar TODAS as Features de Proteínas

In [ ]:
# === COMBINAR TODAS AS FEATURES DE PROTEÍNAS ===

print("\n" + "="*60)
print("COMBINANDO FEATURES DE PROTEÍNAS")
print("="*60)

# TREINO - Começa com AAC (já tem labels)
df_train_final = pd.read_csv(DATA_PROC / "train_features.csv")
print(f"\n1. AAC: {df_train_final.shape}")

# DPC
if 'df_dpc_train' in globals():
    df_dpc_train_clean = df_dpc_train.drop(columns=['label'], errors='ignore')
    df_train_final = df_train_final.merge(df_dpc_train_clean, on='nameseq', how='left')
    print(f"2. + DPC: {df_train_final.shape}")

# Entropy
if 'df_entropy_train' in globals():
    df_entropy_train_clean = df_entropy_train.drop(columns=['label'], errors='ignore')
    df_train_final = df_train_final.merge(df_entropy_train_clean, on='nameseq', how='left')
    print(f"3. + Entropy: {df_train_final.shape}")

# Mappings (opcional - requer input interativo)
if 'df_mappings_train' in globals() and df_mappings_train is not None:
    df_mappings_train_clean = df_mappings_train.drop(columns=['label'], errors='ignore')
    df_train_final = df_train_final.merge(df_mappings_train_clean, on='nameseq', how='left')
    print(f"4. + Protein Mappings: {df_train_final.shape}")
else:
    print("4. Protein Mappings: SKIPPED (requer input interativo)")

# Preenche missing
df_train_final = df_train_final.fillna(0)

# Salva
df_train_final.to_csv(DATA_PROC / "train_features.csv", index=False)
print(f"\nTREINO FINAL: {df_train_final.shape}")

# TEST
df_test_final = pd.read_csv(DATA_PROC / "test_features.csv")
print(f"\n1. AAC (test): {df_test_final.shape}")

if 'df_dpc_test' in globals():
    df_dpc_test_clean = df_dpc_test.drop(columns=['label'], errors='ignore')
    df_test_final = df_test_final.merge(df_dpc_test_clean, on='nameseq', how='left')
    print(f"2. + DPC: {df_test_final.shape}")

if 'df_entropy_test' in globals():
    df_entropy_test_clean = df_entropy_test.drop(columns=['label'], errors='ignore')
    df_test_final = df_test_final.merge(df_entropy_test_clean, on='nameseq', how='left')
    print(f"3. + Entropy: {df_test_final.shape}")

if 'df_mappings_test' in globals() and df_mappings_test is not None:
    df_mappings_test_clean = df_mappings_test.drop(columns=['label'], errors='ignore')
    df_test_final = df_test_final.merge(df_mappings_test_clean, on='nameseq', how='left')
    print(f"4. + Protein Mappings: {df_test_final.shape}")
else:
    print("4. Protein Mappings: SKIPPED")

df_test_final = df_test_final.fillna(0)
df_test_final.to_csv(DATA_PROC / "test_features.csv", index=False)
print(f"\nTEST FINAL: {df_test_final.shape}")

print("\n" + "="*60)
print("RESUMO")
print("="*60)
print(f"Treino: {df_train_final.shape[0]} amostras x {df_train_final.shape[1]-2} features")
print(f"Teste:  {df_test_final.shape[0]} amostras x {df_test_final.shape[1]-1} features")
print("\nFeatures de PROTEÍNAS:")
print("  - AAC (Amino Acid Composition) - 20 features")
print("  - DPC (Dipeptide Composition) - 400 features")
print("  - Entropy Shannon - ~10 features")
print(f"\nTotal: ~430 features específicas para proteínas!")
print("\nPronto para modelagem!")


COMBINANDO FEATURES DE PROTEÍNAS

1. AAC: (3880, 22)
2. + DPC: (3880, 422)
3. + Entropy: (3880, 425)

TREINO FINAL: (3880, 425)

1. AAC (test): (970, 21)
2. + DPC: (970, 421)
3. + Entropy: (970, 424)

TREINO FINAL: (3880, 425)

1. AAC (test): (970, 21)
2. + DPC: (970, 421)
3. + Entropy: (970, 424)

TEST FINAL: (970, 424)

RESUMO
Treino: 3880 amostras x 423 features
Teste:  970 amostras x 423 features

Features de PROTEÍNAS:
  - AAC (Amino Acid Composition)
  - DPC (Dipeptide Composition)
  - Entropy Shannon
  - Protein Mappings (EIIP + Fourier)

Pronto para modelagem!

TEST FINAL: (970, 424)

RESUMO
Treino: 3880 amostras x 423 features
Teste:  970 amostras x 423 features

Features de PROTEÍNAS:
  - AAC (Amino Acid Composition)
  - DPC (Dipeptide Composition)
  - Entropy Shannon
  - Protein Mappings (EIIP + Fourier)

Pronto para modelagem!
